# Awaaz — Colab Checkpoint Averaging, Evaluation, and ONNX Export

This notebook finalizes a trained Awaaz speech recognition model:
1. Performs checkpoint averaging across the top-$N$ saved checkpoints to enhance generalizability.
2. Evaluates Word Error Rate (WER) and Character Error Rate (CER) on bilingual test sets.
3. Exports model to ONNX (opset 17) for both offline and streaming modes.
4. Quantizes ONNX models to int8 for low-latency edge deployment.
5. Packages release artifacts.

In [ ]:
# 1. Mount Google Drive and Setup Environment
from google.colab import drive

drive.mount('/content/drive')

%cd /content
!rm -rf repo
!git clone https://github.com/Paramveersingh-S/Awaaz.git repo
%cd /content/repo
!git checkout main
!pip install -q --upgrade pip
!pip install -e .[export]

In [ ]:
# 2. Average Top-10 Checkpoints for Generalization
!mkdir -p /content/drive/MyDrive/awaaz/release

!python -m awaaz.average \
  --checkpoints /content/drive/MyDrive/awaaz/runs/small_v1/step_*.pt \
  --num_checkpoints 10 \
  --output /content/drive/MyDrive/awaaz/release/awaaz_small_averaged.pt

print('Checkpoint averaging complete.')

In [ ]:
# 3. Bilingual Evaluation (Offline and Streaming-Chunked)
!python -m awaaz.evaluate \
  --config configs/small.yaml \
  --checkpoint /content/drive/MyDrive/awaaz/release/awaaz_small_averaged.pt \
  --manifest /content/drive/MyDrive/awaaz/data/test_combined.jsonl \
  --tokenizer /content/drive/MyDrive/awaaz/artifacts/tokenizer.model \
  --cmvn /content/drive/MyDrive/awaaz/artifacts/cmvn.npz \
  --output_dir /content/drive/MyDrive/awaaz/release/eval_results

In [ ]:
# 4. Export ONNX (Offline & Streaming Windows) and Dynamic Quantization (int8)
!python -m awaaz.export \
  --config configs/small.yaml \
  --checkpoint /content/drive/MyDrive/awaaz/release/awaaz_small_averaged.pt \
  --output_dir /content/drive/MyDrive/awaaz/release/onnx \
  --quantize \
  --opset 17

print('ONNX models (fp32 and int8) successfully exported.')

In [ ]:
# 5. Inspect Release Artifact Footprint
!ls -lh /content/drive/MyDrive/awaaz/release/onnx